In [1]:
# 1. 외부 모듈 자동 새로고침 설정 (loader.py 수정 시 즉각 반영)
%load_ext autoreload
%autoreload 2

# 2. 필수 라이브러리 임포트
import os
import json
import pandas as pd
import FinanceDataReader as fdr
import pykrx
import OpenDartReader
import matplotlib
import seaborn
import scipy
from datetime import date

# 3. 직접 만든 로컬 모듈 임포트
from data.loader import QuantDataLoader

print("✅ 환경 설정 및 전체 라이브러리 정상 로드 완료!")

KRX 로그인 시도...
  로그인 ID: forscom
KRX 로그인 완료.
  로그인 시간: 2026-07-28 11:52:16
  만료 시간: 2026-07-28 12:52:16
✅ 환경 설정 및 전체 라이브러리 정상 로드 완료!


In [3]:
def test_data_loader():
    print("==================================================")
    print("🚀 QuantDataLoader 테스트를 시작합니다...")
    print("==================================================\n")
    
    # 1. 로더 인스턴스 생성
    try:
        print("[테스트 1] 로더 인스턴스화 및 환경변수 확인")
        loader = QuantDataLoader(use_cache=True)
        print("✅ 성공: DART API 키 및 로더 초기화 완료!\n")
    except Exception as e:
        print(f"❌ 실패: {e}\n")
        return

    # 2. 유니버스 로드 테스트 (Point-in-Time)
    test_date = date(2023, 7, 24)
    print(f"[테스트 2] KOSPI 유니버스 데이터 로드 ({test_date})")
    try:
        universe_df = loader.get_kospi_universe(test_date)
        print(f"✅ 성공: 총 {len(universe_df)}개 종목 로드 완료!")
        print("-" * 50)
        display(universe_df.head())
        print("-" * 50 + "\n")
    except Exception as e:
        print(f"❌ 실패: {e}\n")

    # 3. DART 재무제표 파싱 테스트
    ticker_to_test = '005930'
    target_year = 2023
    print(f"[테스트 3] {ticker_to_test} {target_year}년 사업보고서(11011) 파싱")
    try:
        financials = loader.parse_standardized_financials(ticker_to_test, target_year, '11011')
        print(f"✅ 성공: 재무 데이터 표준화 완료!")
        print("-" * 50)
        for key, value in financials.items():
            if pd.isna(value):
                print(f"{key:>20} : NaN")
            else:
                print(f"{key:>20} : {value:,.0f}")
        print("-" * 50 + "\n")
    except Exception as e:
        print(f"❌ 실패: {e}\n")

    print("==================================================")
    print("🎯 모든 테스트가 종료되었습니다.")
    print("==================================================")

# 테스트 실행
test_data_loader()

🚀 QuantDataLoader 테스트를 시작합니다...

[테스트 1] 로더 인스턴스화 및 환경변수 확인
✅ 성공: DART API 키 및 로더 초기화 완료!

[테스트 2] KOSPI 유니버스 데이터 로드 (2023-07-24)
✅ 성공: 총 834개 종목 로드 완료!
--------------------------------------------------


,ticker,name,sector,close_price,market_cap
0,005930,삼성전자,통신 및 방송 장비 제조업,70400,420272691520000
1,373220,LG에너지솔루션,일차전지 및 이차전지 제조업,597000,139698000000000
2,000660,SK하이닉스,반도체 제조업,114000,82992269610000
3,005490,POSCO홀딩스,1차 철강 제조업,642000,54294729660000
4,207940,삼성바이오로직스,기초 의약물질 제조업,742000,52811108000000


--------------------------------------------------

[테스트 3] 005930 2023년 사업보고서(11011) 파싱
✅ 성공: 재무 데이터 표준화 완료!
--------------------------------------------------
             revenue : 258,935,494,000,000
                cogs : 180,388,580,000,000
        gross_profit : 78,546,914,000,000
                 sga : 71,979,938,000,000
           inventory : 51,625,874,000,000
    operating_income : 6,566,976,000,000
          net_income : 15,487,100,000,000
 operating_cash_flow : 44,137,427,000,000
--------------------------------------------------

🎯 모든 테스트가 종료되었습니다.


In [4]:
# config.json 파라미터 불러오기
try:
    with open('config.json', 'r', encoding='utf-8') as f:
        config = json.load(f)
    
    target_market = config['strategy_params']['market'] # 예: 'KOSPI'
    top_n = config['strategy_params']['top_n_mcap']
except FileNotFoundError:
    print("⚠️ config.json 파일이 없습니다. 기본값으로 진행합니다.")
    target_market = 'KOSPI'
    top_n = 10

print(f"🔍 {target_market} 시장 데이터를 불러오는 중...\n")

# FinanceDataReader를 통한 KRX 전종목 리스팅 조회
df_krx = fdr.StockListing('KRX')

# 지정한 시장 필터링 및 시가총액(MarCap) 기준 정렬
top_mcap_df = df_krx[df_krx['Market'] == target_market].sort_values(by='Marcap', ascending=False).head(top_n)

# 보기 좋게 컬럼명 정리
top_mcap_df = top_mcap_df[['Code', 'Name', 'Close', 'Marcap', 'Stocks']].rename(
    columns={
        'Code': '종목코드',
        'Name': '종목명',
        'Close': '종가',
        'Marcap': '시가총액',
        'Stocks': '상장주식수'
    }
)

print("✅ 정상적으로 데이터를 불러왔습니다!")
display(top_mcap_df)

🔍 KOSPI 시장 데이터를 불러오는 중...

✅ 정상적으로 데이터를 불러왔습니다!


,종목코드,종목명,종가,시가총액,상장주식수
0,005930,삼성전자,230000,1344644079840000,5846278608
1,000660,SK하이닉스,1616000,1151727021840000,712702365
2,005935,삼성전자우,164200,131749351532600,802371203
3,402340,SK스퀘어,964000,127207884104000,131958386
4,009150,삼성전기,1141000,85225507136000,74693696
5,005380,현대차,374000,76579404484000,204757766
6,373220,LG에너지솔루션,317500,74295000000000,234000000
7,207940,삼성바이오로직스,1559000,72167592609000,46290951
8,105560,KB금융,170600,60509727420400,354687734
9,032830,삼성생명,286000,57200000000000,200000000


In [1]:
import pandas as pd
from datetime import date
from data.loader import QuantDataLoader

def verify_new_features():
    print("==================================================")
    print("🚀 QuantDataLoader 신규 기능 검증을 시작합니다...")
    print("==================================================\n")
    
    try:
        loader = QuantDataLoader(use_cache=True)
    except Exception as e:
        print(f"❌ 초기화 실패: {e}")
        return

    # ---------------------------------------------------------
    # 검증 1: 시계열 주가/거래량 데이터 (OHLCV) 및 캐싱
    # ---------------------------------------------------------
    print("[검증 1] get_historical_ohlcv 작동 확인")
    ticker = '005930'
    start = date(2025, 1, 1)
    end = date(2025, 6, 30)
    
    try:
        ohlcv_df = loader.get_historical_ohlcv(ticker, start, end)
        if ohlcv_df is not None and not ohlcv_df.empty:
            print(f"✅ 성공: {start} ~ {end} 시계열 데이터 {len(ohlcv_df)}일치 로드 완료")
            print(ohlcv_df[['Close', 'Volume']].head(3).to_string())
        else:
            print("❌ 실패: 데이터가 비어 있습니다.")
    except Exception as e:
        print(f"❌ 실패: {e}")
    print("-" * 50 + "\n")

    # ---------------------------------------------------------
    # 검증 2: 확장된 계정과목 매핑 확인 (자산, 부채, 자본 등)
    # ---------------------------------------------------------
    print("[검증 2] 확장된 재무제표 계정 파싱 확인 (2025년 사업보고서)")
    try:
        fin_annual = loader.parse_standardized_financials(ticker, 2025, '11011')
        keys_to_check = ['total_assets', 'total_liabilities', 'total_equity', 'interest_expense']
        
        missing = [k for k in keys_to_check if pd.isna(fin_annual.get(k, float('nan')))]
        if not missing:
            print("✅ 성공: 자산/부채/자본/이자비용 모두 정상 파싱 완료")
            for k in keys_to_check:
                print(f"   - {k}: {fin_annual[k]:,.0f}")
        else:
            print(f"⚠️ 주의: 다음 계정 누락 (정상일 수도 있음) -> {missing}")
    except Exception as e:
        print(f"❌ 실패: {e}")
    print("-" * 50 + "\n")

    # ---------------------------------------------------------
    # 검증 3: 분기 단독값 차분(Isolation) 로직 확인
    # ---------------------------------------------------------
    print("[검증 3] get_isolated_quarterly_financials 차분 로직 확인")
    try:
        # 1분기(누적)와 2분기(차분)의 매출액 비교
        q1_data = loader.get_isolated_quarterly_financials(ticker, 2025, 1)
        q2_isolated = loader.get_isolated_quarterly_financials(ticker, 2025, 2)
        
        print("✅ 성공: 1분기 및 2분기(단독) 데이터 추출 완료")
        print(f"   - 1Q 매출액 (누적=단독) : {q1_data.get('revenue', 0):,.0f}")
        print(f"   - 2Q 매출액 (차분 적용) : {q2_isolated.get('revenue', 0):,.0f}")
    except Exception as e:
        print(f"❌ 실패: {e}")
    print("-" * 50 + "\n")

    # ---------------------------------------------------------
    # 검증 4: DART API Rate Limit 카운터 작동 확인
    # ---------------------------------------------------------
    print("[검증 4] DART API 카운터 및 Rate Limit 방어벽 확인")
    try:
        current_calls = loader.dart_call_count
        print(f"✅ 성공: 현재 세션 API 호출 횟수 정상 트래킹 중 -> {current_calls}회")
        
        # 임의로 한도를 초과시켜 방어벽 테스트
        loader.dart_daily_limit = current_calls  
        
        # 💡 API를 강제로 호출하도록 일시적으로 캐시 기능 비활성화
        loader.use_cache = False 
        
        try:
            loader.get_financial_statements(ticker, 2024, '11011')
            print("❌ 실패: 한도 초과 상황에서 Exception이 발생하지 않고 통과됨!")
        except Exception as limit_err:
            print(f"✅ 성공: 방어벽 정상 작동 확인 -> {limit_err}")
            
    except Exception as e:
        print(f"❌ 실패: {e}")
    print("==================================================\n")


if __name__ == "__main__":
    verify_new_features()

KRX 로그인 시도...
  로그인 ID: forscom
KRX 로그인 완료.
  로그인 시간: 2026-07-28 13:42:19
  만료 시간: 2026-07-28 14:42:19
🚀 QuantDataLoader 신규 기능 검증을 시작합니다...

[검증 1] get_historical_ohlcv 작동 확인
✅ 성공: 2025-01-01 ~ 2025-06-30 시계열 데이터 118일치 로드 완료
            Close    Volume
Date                       
2025-01-02  53400  16630538
2025-01-03  54400  19318046
2025-01-06  55900  19034284
--------------------------------------------------

[검증 2] 확장된 재무제표 계정 파싱 확인 (2025년 사업보고서)
✅ 성공: 자산/부채/자본/이자비용 모두 정상 파싱 완료
   - total_assets: 566,942,110,000,000
   - total_liabilities: 130,621,773,000,000
   - total_equity: 436,320,337,000,000
   - interest_expense: 11,733,764,000,000
--------------------------------------------------

[검증 3] get_isolated_quarterly_financials 차분 로직 확인
✅ 성공: 1분기 및 2분기(단독) 데이터 추출 완료
   - 1Q 매출액 (누적=단독) : 79,140,503,000,000
   - 2Q 매출액 (차분 적용) : 74,566,317,000,000
--------------------------------------------------

[검증 4] DART API 카운터 및 Rate Limit 방어벽 확인
✅ 성공: 현재 세션 API 호출 횟수 정상 트래킹 중 -> 0회
✅ 성공: